In [3]:
import pandas as pd

# Sahi paths ke sath data load karein (data/raw/processed/ ke andar)
sales_daily = pd.read_csv('../data/raw/processed/sales_daily.csv')
inventory_snapshots = pd.read_csv('../data/raw/processed/inventory_snapshots.csv')
calendar = pd.read_csv('../data/raw/processed/calendar.csv')
sku_master = pd.read_csv('../data/raw/processed/sku_master.csv')

# Data load hone ki pushti karein
display(sales_daily.head())

,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0


In [4]:
# Sales data ka structure aur missing values check karein
print("--- Sales Daily Info ---")
sales_daily.info()
print("\nMissing Values:\n", sales_daily.isnull().sum())

# Inventory data ka structure aur missing values check karein
print("\n--- Inventory Info ---")
inventory_snapshots.info()
print("\nMissing Values:\n", inventory_snapshots.isnull().sum())

# Calendar aur SKU data ke liye bhi same check karein
calendar.info()
sku_master.info()

--- Sales Daily Info ---
<class 'pandas.DataFrame'>
RangeIndex: 36550 entries, 0 to 36549
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Date        36550 non-null  str    
 1   SKU         36550 non-null  str    
 2   Units_Sold  36550 non-null  int64  
 3   Revenue     36550 non-null  float64
 4   Price       36550 non-null  float64
 5   Promotion   36550 non-null  int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 2.2 MB

Missing Values:
 Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64

--- Inventory Info ---
<class 'pandas.DataFrame'>
RangeIndex: 4800 entries, 0 to 4799
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Snapshot_Date    4800 non-null   str    
 1   SKU              4800 non-null   str    
 2   Current_Stock    4800 non-null   int64  
 3   On_Order   

In [5]:
import pandas as pd
import os

def run_pipeline():
    print("Starting Data Pipeline...")
    
    # Paths define karein (Aapke data/raw/processed/ folder ke hisaab se)
    data_dir = 'data/raw/processed/'
    output_file = 'data/analysis_ready_data.csv'
    
    # 1. INGESTION: Charo files load karein
    print("Ingesting data...")
    sales = pd.read_csv(os.path.join(data_dir, 'sales_daily.csv'))
    inventory = pd.read_csv(os.path.join(data_dir, 'inventory_snapshots.csv'))
    calendar = pd.read_csv(os.path.join(data_dir, 'calendar.csv'))
    sku = pd.read_csv(os.path.join(data_dir, 'sku_master.csv'))
    
    # 2. CLEANING: Data types theek karein aur missing values handle karein
    print("Cleaning data...")
    
    # A. Dates ko string se Datetime format mein badlein
    sales['date'] = pd.to_datetime(sales['date'])
    calendar['date'] = pd.to_datetime(calendar['date'])
    inventory['date'] = pd.to_datetime(inventory['date'])
    
    # B. Duplicates hatayein
    sales = sales.drop_duplicates()
    
    # C. Missing values handle karein (Jaise agar promo flag khali hai toh use 0 maan lein)
    if 'promo_flag' in sales.columns:
        sales['promo_flag'] = sales['promo_flag'].fillna(0)
    
    # 3. UNIFY: Ek single dataset banayein
    print("Unifying datasets...")
    # Sales aur SKU ko jodein
    df_merged = pd.merge(sales, sku, on='sku_id', how='left')
    
    # Phir usme Calendar jodein
    df_merged = pd.merge(df_merged, calendar, on='date', how='left')
    
    # 4. OUTPUT: Analysis-ready file save karein
    print("Saving analysis-ready dataset...")
    df_merged.to_csv(output_file, index=False)
    
    print(f"Pipeline finished successfully! Clean data saved to: {output_file}")

if __name__ == "__main__":
    run_pipeline()

Starting Data Pipeline...
Ingesting data...


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/processed/sales_daily.csv'